In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum

In [2]:
spark = SparkSession.builder \
    .appName("NYC_Taxi_Data_Pipeline") \
    .getOrCreate()

In [3]:
df = spark.read.parquet("../data/raw/yellow_tripdata_2014-03.parquet")

In [4]:
df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: integer (nullable = true)
 |-- airport_fee: integer (nullable = true)



In [5]:
df.show(5)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2014-03-01 00:55:41|  2014-03-01 00:57:42|              1|          0.0|         1|                 N|         145|         145|           2|        3.0|  0.5|    0.5|       0.

In [6]:
df.count()

15428134

In [7]:
len(df.columns)

19

In [8]:
df.select(
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "store_and_fwd_flag"
).describe().show()

+-------+------------------+------------------+------------------+------------------+--------------------+-----------+------------------+
|summary|     trip_distance|       fare_amount|        tip_amount|      total_amount|congestion_surcharge|airport_fee|store_and_fwd_flag|
+-------+------------------+------------------+------------------+------------------+--------------------+-----------+------------------+
|  count|          15428134|          15428134|          15428134|          15428134|                   0|          0|           7620823|
|   mean|3.8134811766614884|12.215581225830778|1.4731240686662246|14.761783681786595|                NULL|       NULL|              NULL|
| stddev|1982.6178988411455|10.092548583388256| 2.247891680206382|12.243391462701169|                NULL|       NULL|              NULL|
|    min|               0.0|           -612.42|               0.0|               0.0|                NULL|       NULL|                 N|
|    max|         5005013.0|      

In [9]:
df.select(
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "store_and_fwd_flag"
).summary().show()

+-------+------------------+------------------+------------------+------------------+--------------------+-----------+------------------+
|summary|     trip_distance|       fare_amount|        tip_amount|      total_amount|congestion_surcharge|airport_fee|store_and_fwd_flag|
+-------+------------------+------------------+------------------+------------------+--------------------+-----------+------------------+
|  count|          15428134|          15428134|          15428134|          15428134|                   0|          0|           7620823|
|   mean|3.8134811766614884|12.215581225830778|1.4731240686662246|14.761783681786595|                NULL|       NULL|              NULL|
| stddev|1982.6178988411455|10.092548583388256| 2.247891680206382|12.243391462701169|                NULL|       NULL|              NULL|
|    min|               0.0|           -612.42|               0.0|               0.0|                NULL|       NULL|                 N|
|    25%|              1.01|      

In [10]:
null_counts = df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]
)

null_counts.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|              0|            0|         0|           7807311|           0|           0|           0|          0|    0|      0|         

### Observations

- The dataset contains 15,428,134 taxi trips.
- There are 19 columns.
- Pickup and dropoff timestamps are stored as `timestamp_ntz`.
- `congestion_surcharge` and `airport_fee` contain many NULL values because these fees were not applicable to older trips.
- The dataset appears suitable for further transformation and analysis.

In [11]:
df.filter(col("trip_distance") < 0).count()

0

In [12]:
df.filter(col("total_amount") < 0).count()

0

### I am going to use the following sql functions 
    col,
    sum,
    min,
    max,
    unix_timestamp,
    to_date,
    hour,
    dayofweek,
    when


In [13]:
import pyspark.sql.functions as F

In [14]:
df.select(
    F.min("tpep_pickup_datetime").alias("min_pickup"),
    F.max("tpep_pickup_datetime").alias("max_pickup")
).show()

+-------------------+-------------------+
|         min_pickup|         max_pickup|
+-------------------+-------------------+
|2014-03-01 00:00:00|2014-03-31 23:59:58|
+-------------------+-------------------+



### Data Quality Check Observations

- The dataset contains 15,428,134 taxi trips for March 2014.
- No negative values were found in `trip_distance` and `total_amount`.
- Pickup timestamps range from 2014-03-01 to 2014-03-31.
- `congestion_surcharge` and `airport_fee` contain NULL values for all records because these fields were not applicable for this historical dataset.
- `store_and_fwd_flag` contains approximately 7.8M NULL values and may require handling during transformation.

### Data Transformations

In [15]:
df_transformed = df

In [17]:
df_transformed.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes"
).show(5)

+--------------------+---------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+--------------------+---------------------+---------------------+
| 2014-03-01 00:55:41|  2014-03-01 00:57:42|   2.0166666666666666|
| 2014-03-01 00:24:08|  2014-03-01 00:33:23|                 9.25|
| 2014-03-01 00:44:43|  2014-03-01 00:59:32|   14.816666666666666|
| 2014-03-01 00:11:32|  2014-03-01 00:26:48|   15.266666666666667|
| 2014-03-01 00:29:39|  2014-03-01 00:39:36|                 9.95|
+--------------------+---------------------+---------------------+
only showing top 5 rows


In [18]:
df_transformed = df_transformed.withColumn(
    "pickup_date",
    F.to_date("tpep_pickup_datetime")
)

In [19]:
df_transformed.select(
    "tpep_pickup_datetime",
    "pickup_date"
).show(5)

+--------------------+-----------+
|tpep_pickup_datetime|pickup_date|
+--------------------+-----------+
| 2014-03-01 00:55:41| 2014-03-01|
| 2014-03-01 00:24:08| 2014-03-01|
| 2014-03-01 00:44:43| 2014-03-01|
| 2014-03-01 00:11:32| 2014-03-01|
| 2014-03-01 00:29:39| 2014-03-01|
+--------------------+-----------+
only showing top 5 rows


 ### `trip_duration_minutes` and `pickup_date` дают возможность делать аналитику по:

- длительности поездок;
- дням;
- периодам;
- трендам.

In [20]:
df_transformed = df_transformed.withColumn(
    "pickup_hour",
    F.hour("tpep_pickup_datetime")
)

In [21]:
df_transformed = df_transformed.withColumn(
    "pickup_day_of_week",
    F.dayofweek("tpep_pickup_datetime")
)

In [22]:
df_transformed = df_transformed.withColumn(
    "avg_speed_mph",
    F.when(
        col("trip_duration_minutes") > 0,
        col("trip_distance") / (col("trip_duration_minutes") / 60)
    )
)

 #### `pickup_hour`, `pickup_day_of_week`, and `avg_speed_mph` 
 ### дают возможность делать аналитику по:

- в какие часы больше всего поездок?;
- анализ будни/выходные;
- скорость поездок.

In [23]:
df_transformed.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: integer (nullable = true)
 |-- airport_fee: integer (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = t

### Handle NULL Values

We handle NULL values because missing data can affect analysis results and cause problems during calculations and aggregations.

In this dataset:
- `store_and_fwd_flag` contains NULL values, so we will replace them with `"Unknown"`.
- `congestion_surcharge` and `airport_fee` contain only NULL values, so we will remove these columns.

In [24]:
df_transformed = df_transformed.fillna(
    {"store_and_fwd_flag": "Unknown"}
)

In [25]:
df_transformed.filter(
    F.col("store_and_fwd_flag").isNull()
).count()

0

In [26]:
df_transformed = df_transformed.drop(
    "congestion_surcharge",
    "airport_fee"
)

In [27]:
df_transformed.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = false)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_day_of_week: integer (nullable = true)
 |-- avg_speed_mph: double (nullable = t

In [28]:
df_transformed.select(
    "store_and_fwd_flag"
).groupBy(
    "store_and_fwd_flag"
).count().show()

+------------------+-------+
|store_and_fwd_flag|  count|
+------------------+-------+
|                 Y| 175137|
|           Unknown|7807311|
|                 N|7445686|
+------------------+-------+



### Select Analytical Columns

We select the columns needed for analysis.
The analytical dataset contains original business fields and newly created features.

In [29]:
df_analytics = df_transformed.select(
    "VendorID",
    "pickup_date",
    "pickup_hour",
    "pickup_day_of_week",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "avg_speed_mph",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "payment_type",
    "store_and_fwd_flag"
)

In [30]:
df_analytics.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_day_of_week: integer (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- avg_speed_mph: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = false)



In [31]:
df_analytics.show(5)


+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+------------------+-----------+----------+------------+------------+------------------+
|VendorID|pickup_date|pickup_hour|pickup_day_of_week|PULocationID|DOLocationID|passenger_count|trip_distance|trip_duration_minutes|     avg_speed_mph|fare_amount|tip_amount|total_amount|payment_type|store_and_fwd_flag|
+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+------------------+-----------+----------+------------+------------+------------------+
|       1| 2014-03-01|          0|                 7|         145|         145|              1|          0.0|   2.0166666666666666|               0.0|        3.0|       0.0|         4.0|           2|                 N|
|       1| 2014-03-01|          0|                 7|         237|          48|              3|          1.3|               

### Final Data Quality Check

In [32]:
df_analytics.count()

15428134

In [33]:
df_analytics.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_day_of_week: integer (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- avg_speed_mph: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = false)



In [34]:
df_analytics.select(
    [
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_analytics.columns
    ]
).show()

+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+-------------+-----------+----------+------------+------------+------------------+
|VendorID|pickup_date|pickup_hour|pickup_day_of_week|PULocationID|DOLocationID|passenger_count|trip_distance|trip_duration_minutes|avg_speed_mph|fare_amount|tip_amount|total_amount|payment_type|store_and_fwd_flag|
+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+-------------+-----------+----------+------------+------------+------------------+
|       0|          0|          0|                 0|           0|           0|              0|            0|                    0|        42416|          0|         0|           0|           0|                 0|
+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+-------------

- `avg_speed_mph` contains NULL values for trips with zero duration because speed cannot be calculated.

### Final Data Quality Check

The analytical dataset was validated after transformation.

- The number of records remained unchanged after transformations.
- Required analytical columns were created successfully.
- NULL values were handled for selected columns.
- `avg_speed_mph` contains NULL values (in 42416 rows) for trips with zero duration because speed cannot be calculated.
- The dataset is ready for downstream analytics.

### Spark Internals: Partitions

Spark divides data into partitions and processes partitions in parallel.
The number of partitions affects parallelism and performance.

In [35]:
df_analytics.rdd.getNumPartitions()

12

### Spark Internals: Lazy Evaluation and Explain Plan

Spark uses lazy evaluation. Transformations are not executed immediately.
Spark builds an execution plan and runs the computation only when an action is called.

In [36]:
df_test = df_analytics.select(
    "pickup_date",
    "total_amount"
)

In [37]:
df_test.explain()

== Physical Plan ==
*(1) Project [cast(tpep_pickup_datetime#1 as date) AS pickup_date#1927, total_amount#16]
+- *(1) ColumnarToRow
   +- FileScan parquet [tpep_pickup_datetime#1,total_amount#16] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Murch24/aws-pyspark-data-engineering-project/data/raw/y..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,total_amount:double>




In [38]:
df_test.explain(True)

== Parsed Logical Plan ==
'Project ['pickup_date, 'total_amount]
+- Project [VendorID#0L, pickup_date#1927, pickup_hour#1935, pickup_day_of_week#1936, PULocationID#7L, DOLocationID#8L, passenger_count#3L, trip_distance#4, trip_duration_minutes#1916, avg_speed_mph#1937, fare_amount#10, tip_amount#13, total_amount#16, payment_type#9L, store_and_fwd_flag#1938]
   +- Project [VendorID#0L, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#1938, PULocationID#7L, DOLocationID#8L, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, trip_duration_minutes#1916, pickup_date#1927, pickup_hour#1935, pickup_day_of_week#1936, avg_speed_mph#1937]
      +- Project [VendorID#0L, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, coalesce(store_and_fwd_flag#6, cast(Unknown as string)) AS store_and_fwd_flag#193

### Spark Internals: Shuffle

Shuffle happens when Spark needs to redistribute data across partitions.
Operations such as `groupBy`, `join`, `orderBy`, and `distinct` can trigger shuffle.

Shuffle is expensive because it requires data movement between partitions.

In [39]:
df_daily_trips = df_analytics.groupBy(
    "pickup_date"
).count()

In [40]:
df_daily_trips.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[pickup_date#1927], functions=[count(1)])
   +- Exchange hashpartitioning(pickup_date#1927, 200), ENSURE_REQUIREMENTS, [plan_id=645]
      +- HashAggregate(keys=[pickup_date#1927], functions=[partial_count(1)])
         +- Project [cast(tpep_pickup_datetime#1 as date) AS pickup_date#1927]
            +- FileScan parquet [tpep_pickup_datetime#1] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Murch24/aws-pyspark-data-engineering-project/data/raw/y..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz>




In [41]:
spark.conf.get("spark.sql.shuffle.partitions")

'200'

In [42]:
spark.conf.set("spark.sql.shuffle.partitions", "200")

In [43]:
df_daily_trips = df_analytics.groupBy(
    "pickup_date"
).count()

In [44]:
df_daily_trips.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[pickup_date#1927], functions=[count(1)])
   +- Exchange hashpartitioning(pickup_date#1927, 200), ENSURE_REQUIREMENTS, [plan_id=660]
      +- HashAggregate(keys=[pickup_date#1927], functions=[partial_count(1)])
         +- Project [cast(tpep_pickup_datetime#1 as date) AS pickup_date#1927]
            +- FileScan parquet [tpep_pickup_datetime#1] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Murch24/aws-pyspark-data-engineering-project/data/raw/y..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz>




In [45]:
df_daily_trips.show(5)

+-----------+------+
|pickup_date| count|
+-----------+------+
| 2014-03-21|516535|
| 2014-03-05|508381|
| 2014-03-09|473451|
| 2014-03-02|462065|
| 2014-03-30|462771|
+-----------+------+
only showing top 5 rows


### Broadcast Join Observation

Spark automatically selected BroadcastHashJoin because the taxi zone lookup table is small enough to fit in memory.

The execution plan shows `BroadcastExchange`, which means Spark broadcasted the lookup table instead of shuffling the large trip dataset.

Using broadcast join reduces data movement and improves join performance.

In [46]:
df_zones = spark.read.csv(
    "../data/raw/taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)

In [47]:
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [48]:
df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [49]:
df_zones.count()

265

In [50]:
df_join_normal = df_analytics.join(
    df_zones,
    df_analytics.PULocationID == df_zones.LocationID,
    "left"
)

df_join_normal.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [PULocationID#7L], [cast(LocationID#2238 as bigint)], LeftOuter, BuildRight, false, false
   :- Project [VendorID#0L, pickup_date#1927, pickup_hour#1935, pickup_day_of_week#1936, PULocationID#7L, DOLocationID#8L, passenger_count#3L, trip_distance#4, trip_duration_minutes#1916, CASE WHEN (trip_duration_minutes#1916 > 0.0) THEN (trip_distance#4 / (trip_duration_minutes#1916 / 60.0)) END AS avg_speed_mph#1937, fare_amount#10, tip_amount#13, total_amount#16, payment_type#9L, coalesce(store_and_fwd_flag#6, Unknown) AS store_and_fwd_flag#1938]
   :  +- Project [VendorID#0L, passenger_count#3L, trip_distance#4, store_and_fwd_flag#6, PULocationID#7L, DOLocationID#8L, payment_type#9L, fare_amount#10, tip_amount#13, total_amount#16, (cast((unix_timestamp(tpep_dropoff_datetime#2, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true) - unix_timestamp(tpep_pickup_datetime#1, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true)) a

In [51]:
from pyspark.sql.functions import broadcast

df_join_broadcast = df_analytics.join(
    broadcast(df_zones),
    df_analytics.PULocationID == df_zones.LocationID,
    "left"
)

df_join_broadcast.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [PULocationID#7L], [cast(LocationID#2238 as bigint)], LeftOuter, BuildRight, false, false
   :- Project [VendorID#0L, pickup_date#1927, pickup_hour#1935, pickup_day_of_week#1936, PULocationID#7L, DOLocationID#8L, passenger_count#3L, trip_distance#4, trip_duration_minutes#1916, CASE WHEN (trip_duration_minutes#1916 > 0.0) THEN (trip_distance#4 / (trip_duration_minutes#1916 / 60.0)) END AS avg_speed_mph#1937, fare_amount#10, tip_amount#13, total_amount#16, payment_type#9L, coalesce(store_and_fwd_flag#6, Unknown) AS store_and_fwd_flag#1938]
   :  +- Project [VendorID#0L, passenger_count#3L, trip_distance#4, store_and_fwd_flag#6, PULocationID#7L, DOLocationID#8L, payment_type#9L, fare_amount#10, tip_amount#13, total_amount#16, (cast((unix_timestamp(tpep_dropoff_datetime#2, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true) - unix_timestamp(tpep_pickup_datetime#1, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true)) a

In [52]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'10485760b'

In [53]:
df_borough_trips = (
    df_analytics.join(
        F.broadcast(df_zones),
        df_analytics.PULocationID == df_zones.LocationID,
        "left"
    )
    .groupBy("Borough")
    .count()
    .orderBy(F.desc("count"))
)

df_borough_trips.show()

+-------------+--------+
|      Borough|   count|
+-------------+--------+
|    Manhattan|14091296|
|       Queens|  662086|
|     Brooklyn|  396668|
|      Unknown|  254188|
|          N/A|   12112|
|        Bronx|   10741|
|          EWR|     831|
|Staten Island|     212|
+-------------+--------+



### Broadcast Join Example

The taxi trip dataset was joined with the taxi zone lookup table using a broadcast join.

This allowed the analytical dataset to include borough names instead of only location IDs.

The joined dataset was then aggregated to calculate the total number of trips by borough.

In [54]:
df_borough_trips.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Exchange (8)
            +- HashAggregate (7)
               +- Project (6)
                  +- BroadcastHashJoin LeftOuter BuildRight (5)
                     :- Scan parquet  (1)
                     +- BroadcastExchange (4)
                        +- Filter (3)
                           +- Scan csv  (2)


(1) Scan parquet 
Output [1]: [PULocationID#7L]
Batched: true
Location: InMemoryFileIndex [file:/C:/Users/Murch24/aws-pyspark-data-engineering-project/data/raw/yellow_tripdata_2014-03.parquet]
ReadSchema: struct<PULocationID:bigint>

(2) Scan csv 
Output [2]: [LocationID#2238, Borough#2239]
Batched: false
Location: InMemoryFileIndex [file:/C:/Users/Murch24/aws-pyspark-data-engineering-project/data/raw/taxi_zone_lookup.csv]
PushedFilters: [IsNotNull(LocationID)]
ReadSchema: struct<LocationID:int,Borough:string>

(3) Filter
Input [2]: [LocationID#2238, Borough#2239]
Co

In [55]:
df_analytics.groupBy("PULocationID") \
    .count() \
    .orderBy(
        F.desc("count")
    ) \
    .show(10)

+------------+------+
|PULocationID| count|
+------------+------+
|         237|540645|
|         161|539732|
|         162|533056|
|          79|531133|
|         230|524709|
|         170|513206|
|         234|508655|
|          48|503733|
|         236|481162|
|         186|469191|
+------------+------+
only showing top 10 rows


In [56]:
df_analytics.explain()

== Physical Plan ==
*(1) Project [VendorID#0L, pickup_date#1927, pickup_hour#1935, pickup_day_of_week#1936, PULocationID#7L, DOLocationID#8L, passenger_count#3L, trip_distance#4, trip_duration_minutes#1916, CASE WHEN (trip_duration_minutes#1916 > 0.0) THEN (trip_distance#4 / (trip_duration_minutes#1916 / 60.0)) END AS avg_speed_mph#1937, fare_amount#10, tip_amount#13, total_amount#16, payment_type#9L, coalesce(store_and_fwd_flag#6, Unknown) AS store_and_fwd_flag#1938]
+- *(1) Project [VendorID#0L, passenger_count#3L, trip_distance#4, store_and_fwd_flag#6, PULocationID#7L, DOLocationID#8L, payment_type#9L, fare_amount#10, tip_amount#13, total_amount#16, (cast((unix_timestamp(tpep_dropoff_datetime#2, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true) - unix_timestamp(tpep_pickup_datetime#1, yyyy-MM-dd HH:mm:ss, Some(America/Tijuana), true)) as double) / 60.0) AS trip_duration_minutes#1916, cast(tpep_pickup_datetime#1 as date) AS pickup_date#1927, hour(tpep_pickup_datetime#1, Some(America/

In [57]:
df_analytics.cache()

DataFrame[VendorID: bigint, pickup_date: date, pickup_hour: int, pickup_day_of_week: int, PULocationID: bigint, DOLocationID: bigint, passenger_count: bigint, trip_distance: double, trip_duration_minutes: double, avg_speed_mph: double, fare_amount: double, tip_amount: double, total_amount: double, payment_type: bigint, store_and_fwd_flag: string]

In [58]:
df_analytics.count()

15428134

In [59]:
df_test = df_analytics.groupBy("pickup_date").count()

In [60]:
df_test.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (10)
+- HashAggregate (9)
   +- Exchange (8)
      +- HashAggregate (7)
         +- InMemoryTableScan (1)
               +- InMemoryRelation (2)
                     +- * Project (6)
                        +- * Project (5)
                           +- * ColumnarToRow (4)
                              +- Scan parquet  (3)


(1) InMemoryTableScan
Output [1]: [pickup_date#1927]
Arguments: [pickup_date#1927]

(2) InMemoryRelation
Arguments: [VendorID#0L, pickup_date#1927, pickup_hour#1935, pickup_day_of_week#1936, PULocationID#7L, DOLocationID#8L, passenger_count#3L, trip_distance#4, trip_duration_minutes#1916, avg_speed_mph#1937, fare_amount#10, tip_amount#13, total_amount#16, payment_type#9L, store_and_fwd_flag#1938], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan parquet 
Output [12]: [VendorID#0L, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, store_and_fwd_flag#6, PULocationID#7L, DOLocation

In [61]:
spark.version

'4.2.0'

In [62]:
df_analytics.rdd.getNumPartitions()

12

In [63]:
df_analytics.limit(5).show()

+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+------------------+-----------+----------+------------+------------+------------------+
|VendorID|pickup_date|pickup_hour|pickup_day_of_week|PULocationID|DOLocationID|passenger_count|trip_distance|trip_duration_minutes|     avg_speed_mph|fare_amount|tip_amount|total_amount|payment_type|store_and_fwd_flag|
+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+------------------+-----------+----------+------------+------------+------------------+
|       1| 2014-03-01|          0|                 7|         145|         145|              1|          0.0|   2.0166666666666666|               0.0|        3.0|       0.0|         4.0|           2|                 N|
|       1| 2014-03-01|          0|                 7|         237|          48|              3|          1.3|               

In [64]:
df_test = df_analytics.groupBy("pickup_date").count()
df_test.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (10)
+- HashAggregate (9)
   +- Exchange (8)
      +- HashAggregate (7)
         +- InMemoryTableScan (1)
               +- InMemoryRelation (2)
                     +- * Project (6)
                        +- * Project (5)
                           +- * ColumnarToRow (4)
                              +- Scan parquet  (3)


(1) InMemoryTableScan
Output [1]: [pickup_date#1927]
Arguments: [pickup_date#1927]

(2) InMemoryRelation
Arguments: [VendorID#0L, pickup_date#1927, pickup_hour#1935, pickup_day_of_week#1936, PULocationID#7L, DOLocationID#8L, passenger_count#3L, trip_distance#4, trip_duration_minutes#1916, avg_speed_mph#1937, fare_amount#10, tip_amount#13, total_amount#16, payment_type#9L, store_and_fwd_flag#1938], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan parquet 
Output [12]: [VendorID#0L, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, store_and_fwd_flag#6, PULocationID#7L, DOLocation

In [65]:
df_analytics.unpersist()

DataFrame[VendorID: bigint, pickup_date: date, pickup_hour: int, pickup_day_of_week: int, PULocationID: bigint, DOLocationID: bigint, passenger_count: bigint, trip_distance: double, trip_duration_minutes: double, avg_speed_mph: double, fare_amount: double, tip_amount: double, total_amount: double, payment_type: bigint, store_and_fwd_flag: string]

In [66]:
df_analytics.storageLevel

StorageLevel(False, False, False, False, 1)

In [67]:
from pyspark import StorageLevel


In [68]:

df_analytics.persist(StorageLevel.MEMORY_AND_DISK)

DataFrame[VendorID: bigint, pickup_date: date, pickup_hour: int, pickup_day_of_week: int, PULocationID: bigint, DOLocationID: bigint, passenger_count: bigint, trip_distance: double, trip_duration_minutes: double, avg_speed_mph: double, fare_amount: double, tip_amount: double, total_amount: double, payment_type: bigint, store_and_fwd_flag: string]

In [68]:
df_analytics.limit(5).show()

+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+------------------+-----------+----------+------------+------------+------------------+
|VendorID|pickup_date|pickup_hour|pickup_day_of_week|PULocationID|DOLocationID|passenger_count|trip_distance|trip_duration_minutes|     avg_speed_mph|fare_amount|tip_amount|total_amount|payment_type|store_and_fwd_flag|
+--------+-----------+-----------+------------------+------------+------------+---------------+-------------+---------------------+------------------+-----------+----------+------------+------------+------------------+
|       1| 2014-03-01|          0|                 7|         145|         145|              1|          0.0|   2.0166666666666666|               0.0|        3.0|       0.0|         4.0|           2|                 N|
|       1| 2014-03-01|          0|                 7|         237|          48|              3|          1.3|               

In [74]:
df_raw_2024 = spark.read.parquet(
    "../data/raw/yellow_tripdata_2024-01.parquet",
    "../data/raw/yellow_tripdata_2024-02.parquet",
    "../data/raw/yellow_tripdata_2024-03.parquet"
)

In [75]:
df_raw_2024.count()

9554778

In [76]:
df_raw_2024.inputFiles()

['file:///C:/Users/Murch24/aws-pyspark-data-engineering-project/data/raw/yellow_tripdata_2024-01.parquet',
 'file:///C:/Users/Murch24/aws-pyspark-data-engineering-project/data/raw/yellow_tripdata_2024-02.parquet',
 'file:///C:/Users/Murch24/aws-pyspark-data-engineering-project/data/raw/yellow_tripdata_2024-03.parquet']

In [77]:
df_raw_2024.rdd.getNumPartitions()

12

In [81]:
df_raw_2024.explain()

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it